# 임의의 타겟 테스트보드에 Husky를 통한 부채널 신호 수집 방안 연구
> 공격 시나리오 : Husky을 활용하여 칩위스퍼러 라이트에 와이어테핑 후 부채널 신호 수집
> - 단, 트리거 신호는 칩위스퍼러 라이트의 소스코드에 존재함

In [1]:
%run My_script.ipynb

Loading BokehJS ...

In [2]:
import chipwhisperer as cw

PLATFORM='CW308_STM32F3'
SCOPETYPE = 'OPENADC'
CRYPTO_TARGET='NONE'
SS_VER='SS_VER_2_1'

# ─────────────────────────────────────────
# ChipWhisperer 다중 장치 연결 관리자
# ─────────────────────────────────────────

def connect_all_devices() -> dict:
    """연결된 모든 ChipWhisperer 장치에 접속하여 딕셔너리로 반환"""

    device_list = cw.list_devices()

    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다.")

    print(f"발견된 장치 수: {len(device_list)}\n")

    scopes = {}

    for device in device_list:
        name = device['name'].replace("-", "_")
        sn   = device['sn']

        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료  (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패  (SN: {sn})\n      └─ {e}")

    return scopes


def disconnect_all_devices(scopes: dict) -> None:
    """딕셔너리 내 모든 장치 연결 해제"""

    print()

    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")

    scopes.clear()


# ─────────────────────────────────────────
# 실행
# ─────────────────────────────────────────

scopes = connect_all_devices()

# 사용 예시
# scopes["ChipWhisperer_Husky"].arm()
# scopes["ChipWhisperer_Lite"].arm()

# 작업 완료 후 연결 해제
# disconnect_all_devices(scopes)

발견된 장치 수: 2

  [✓] ChipWhisperer_Husky 연결 완료  (SN: 502032204c5846303130313137313032)
  [✓] ChipWhisperer_Lite 연결 완료  (SN: 44203120394d36433130322030313035)


In [3]:
if SS_VER == "SS_VER_2_1":
    target_type = cw.targets.SimpleSerial2
else:
    raise OSError("Use SS_VER_2_1")

try:
    target = cw.target(scopes["ChipWhisperer_Lite"], target_type)
    print("[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공")
except:
    print("[✗] ChipWhisperer_Lite에 타겟 보드 연결 실패")

[✓] ChipWhisperer_Lite에 타겟 보드 연결 성공


In [4]:
%%capture
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd simpleserial_main/
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3

In [5]:
if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    raise OSError("Use CW308_STM32F3")

scopes["ChipWhisperer_Lite"].default_setup()

try:
    cw.program_target(scopes["ChipWhisperer_Lite"], prog, f'simpleserial_main/simpleserial-base-{PLATFORM}.hex')
except:
    print("[✗] CW308_STM32F3 타겟 보드에 프로그램 실패")

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 1916844166                to 1959898029               
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 96000000                  to 29538459                 
scope.clock.adc_rate                     changed from 96000000.0                to 29538459.0               
scope.clock.clkgen_

In [6]:
%%capture
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd simpleserial_main/
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 clean

In [7]:
MAX_DATA_LEN = 50  # 한 번에 전송 가능한 최대 데이터 크기 (바이트)

# 재현성을 위해 시드 고정
random.seed(1)

# 무작위 키(key) 및 평문(plaintext) 데이터 생성
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN)
)
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])  
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
print(f'타겟 결과  : {Return_k_XOR_p.hex(" ")} ...')
print(f'골든 모델  : {Golden_k_XOR_p.hex(" ")} ...')
print()

if Golden_k_XOR_p == Return_k_XOR_p:
    print('✅ 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
else:
    print('❌ 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')

=== 결과 비교 ===
타겟 결과  : 55 d5 fe f2 29 be 4a 7d 47 d0 ce 5d 0e 60 fb eb 78 63 a9 6b 58 5d 79 02 a5 18 17 be c5 0b 74 75 74 34 19 31 88 fd 6f 55 8c 6f 8b 7e 69 61 32 7b f1 bc ...
골든 모델  : 55 d5 fe f2 29 be 4a 7d 47 d0 ce 5d 0e 60 fb eb 78 63 a9 6b 58 5d 79 02 a5 18 17 be c5 0b 74 75 74 34 19 31 88 fd 6f 55 8c 6f 8b 7e 69 61 32 7b f1 bc ...

✅ 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)


In [8]:
disconnect_all_devices(scopes)


  [✓] ChipWhisperer_Husky 연결 해제 완료
  [✓] ChipWhisperer_Lite 연결 해제 완료
